# Example code for reproducibility

## Author: PS Kulkarni

### Setup:
1. Load libraries
    - `numpy, pandas, random, seaborn, matplotlib,` etc.
    - `ab_sim` is the local library where we are storing all the HPAI modeling related functions. Make sure you load this one.
2.  The input data is divided in 2 sections:
    - `initial_inputs_all.pkl`: All the basic inputs like transmission rate (`T_low, T_high`), recovery rate (`R_low, R_high`), incubation period (`I_low, I_high`), initially infectious cows (`IC_low, IC_high`)
    - `case_inputs_all.pkl`: All specific inputs for farms: `Farm1:Farm5`
3. Run simulation using `run_main_simulation()` function. This function is in `ab_sim` library which is loaded as `sim_mod` in this example. This function has the following attributes:
    - burn_in_farm_state: The input for the created farm after 10 years of burn-in simulation. The input can be found in `case_inputs_all.pkl` file. 
    - merge_dwell_df: Input for the dwelling time ranges for cows in each pen. The input can be found in `ab_sim/ sim_mod` library under `sim_mod.merge_dwell`
    - ndays: number of days to simulate 
    - burn_in_days: How long the burn-in simulation was performed, default is 3650 days or 10 years
    - initial_infected: Input for `IC` Initially infected cows introduced at start of simulation
    - transmission_rate: Input for `T` Transmission rate parameter 
    - recovery_rate: Input for `R`, Recovery rate parameter
    - incubation_period: Input for `I` Incubation Period 
    - pens_to_mask: If the farm keeps calves offsite, use `['Pen1', 'Pen2', 'Pen3', 'Pen4']`
    - num_simulations: Number of simulations to perform
4. Explore SEIR Curve, Pen counts, lactation characteristics of cows, etc.

In [ ]:
import os
import pandas as pd
import numpy as np
import random
import warnings
warnings.filterwarnings(action = "ignore")
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import copy
import ab_sim as sim_mod

In [ ]:
with open("initial_inputs_all.pkl", "rb") as input1:
    initial_inputs = pickle.load(input1)

initial_inputs

In [ ]:
with open("case_inputs_all.pkl", "rb") as input2:
    farm1 = pickle.load(input2)
farm1 = farm1['Farm1']
farm1.keys()

In [ ]:
sim_list = sim_mod.run_main_simulation(burn_in_farm_state = farm1['burn_in_farm'], 
                                      merge_dwell_df = sim_mod.merge_dwell, 
                                      ndays = 365, 
                                      burn_in_days = 3650, 
                                      initial_infected = farm1['initial_infected'], 
                                      transmission_rate = farm1['T'], 
                                      recovery_rate = farm1['R'], 
                                      incubation_period = farm1['Incubation'], 
                                      pens_to_mask=['Pen1', 'Pen2', 'Pen3', 'Pen4'],
                                      num_simulations=100)

In [ ]:
summary_list = []
for df in sim_list['seir_history']:
    alive_counts = (
        df[df['alive']]
        .groupby(['day', 'pen'])
        .size()
        .reset_index(name='count')
    )
    summary_list.append(alive_counts)

# Concatenate and make a multi-indexed DataFrame
all_runs = pd.concat(summary_list, keys=range(len(summary_list)), names=['sim', 'idx'])

# Pivot to have sims as columns (for aggregation)
all_runs_pivot = all_runs.pivot_table(index=['day', 'pen'], columns='sim', values='count').fillna(0)

day_pen = all_runs[['day', 'pen']].drop_duplicates()

# For each combination, get the counts across all sims
results = []
for _, row in day_pen.iterrows():
    day = row['day']
    pen = row['pen']
    # print(f"Pen: {pen}, Day: {day}, Counts: {vals}")
    # All runs for this day/pen
    vals = all_runs[(all_runs['day'] == day) & (all_runs['pen'] == pen)]['count'].values
    median = np.median(vals)
    q1 = np.percentile(vals, 25)
    q3 = np.percentile(vals, 75)
    iqr = q3 - q1
    results.append({'pen': pen, 'day': day, 'median': median, 'iqr': iqr})

summary_df = pd.DataFrame(results)

summary_df

In [ ]:
pen_labels = pd.DataFrame({'pen': [f'Pen{i}' for i in range(1, 16)],
                        'label' : ["Prewean",
                                  "Postwean",
                                  "Breeding",
                                  "Pregnant",
                                  "Springer",
                                  "Fresh_p",
                                  "Fresh_m",
                                  "High_milk_p",
                                  "Low_milk_p",
                                  "High_milk_m",
                                  "Low_milk_m",
                                  "Dry_cow",
                                  "Close_up",
                                  "Calving",
                                  "Hospital"]})
pen_labels

In [ ]:
summary_df['pen'] = summary_df['pen'].astype(str)
ordered_pens = [f'Pen{i}' for i in range(1, 15+1)]

# Get label mapping from pen_labels (assumes 'pen'/'label' in pen_labels)
pen_to_label = dict(zip(pen_labels['pen'], pen_labels['label']))
ordered_labels = [pen_to_label[pen] for pen in ordered_pens]


In [ ]:
palette = sns.color_palette('Set2', n_colors=15) 
color_dict = {label: palette[i % len(palette)] for i, label in enumerate(ordered_labels)}

plt.figure(figsize=(16, 8))
for pen, label in zip(ordered_pens, ordered_labels):
    group = summary_df[summary_df['pen'] == pen]
    if not group.empty:
        color = color_dict.get(label, 'gray')
        plt.plot(group['day'], group['median'], label=label, color=color)
        plt.fill_between(group['day'],
                         group['median'] - group['iqr'] / 2,
                         group['median'] + group['iqr'] / 2,
                         color=color, alpha=0.2)

plt.xlabel('Day')
plt.ylabel('Number ofCows')
# Legend: Only show labels in pen order
handles, legend_labels = plt.gca().get_legend_handles_labels()
# Reorder legend according to ordered_labels
legend_order = [ordered_labels.index(lbl) for lbl in legend_labels if lbl in ordered_labels]
handles = [handles[i] for i in np.argsort(legend_order)]
labels = [legend_labels[i] for i in np.argsort(legend_order)]
plt.legend(handles, labels, title='Pen Label', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid()  
plt.tight_layout()
# plt.savefig("pen_counts.tiff", dpi = 300)
plt.show()

In [ ]:
seir_summary_list = []
for df in sim_list['seir_history']:
    # Only include alive cows
    df_alive = df[df['alive']]
    # Count per state per day
    counts = df_alive.groupby(['day', 'state']).size().reset_index(name='count')
    seir_summary_list.append(counts)

seir_summary_list[1]

In [ ]:
all_counts = pd.concat(seir_summary_list, keys=range(len(seir_summary_list)), names=['sim', 'idx'])
# For easier aggregation:
pivoted = all_counts.pivot_table(index=['day', 'state'], columns='sim', values='count', fill_value=0)

# Compute stats for each state and day
results = []
for (day, state), vals in pivoted.iterrows():
    med = np.median(vals)
    q1 = np.percentile(vals, 25)
    q3 = np.percentile(vals, 75)
    iqr = q3 - q1
    results.append({'day': day, 'state': state, 'median': med, 'iqr': iqr})
states_summary = pd.DataFrame(results)
states_summary.head()

In [ ]:
state_palette = {'Susceptible': '#1f77b4', 'Exposed': '#ff7f0e', 'Infectious': '#d62728', 'Recovered': '#2ca02c'}

plt.figure(figsize=(12, 7))
for state in ['Susceptible', 
              'Exposed',
              'Infectious',
              'Recovered']:
    g = states_summary[states_summary['state'] == state]
    g = g.sort_values('day')
    plt.plot(g['day'], g['median'], label=state, color=state_palette.get(state))
    plt.fill_between(g['day'],
                     g['median'] - g['iqr'] / 2,
                     g['median'] + g['iqr'] / 2,
                     color=state_palette.get(state, 'gray'),
                     alpha=0.25)

plt.xlabel('Day')
plt.ylabel('Number of Cows (Alive, by SEIR State)')
plt.title('SEIR State Counts by Day (Median, IQR shaded)')
plt.legend(title='SEIR State')
plt.grid()  
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.lines import Line2D

# Concatenate calving intervals, as before
ci_lactation_list = []
for df in sim_list['calving_intervals']:
    valid = df[df['calv_interval'].notnull()]
    ci_lactation_list.append(valid[['lactation_number', 'calv_interval']])

all_ci = pd.concat(ci_lactation_list, ignore_index=True)

# Remove lactation_number == 0
all_ci = all_ci[all_ci['lactation_number'] != 0]

# Plot boxplots
plt.figure(figsize=(12, 6))
ax = sns.boxplot(
    x='lactation_number',
    y='calv_interval',
    hue = 'lactation_number',
    palette = 'Blues',
    data=all_ci,
    showmeans=True,
    meanprops={"marker":"o", "markerfacecolor":"white", 
               "markeredgecolor":"black", "markersize":8},
    showfliers=False
)

plt.xlabel('Lactation Number')
plt.ylabel('Calving Interval (days)')

# Custom legend handles:
mean_legend = Line2D([0], [0], marker='o', color='w',
                      markerfacecolor='white', markeredgecolor='black',
                      markersize=8, label='Mean')
box_legend = Line2D([0], [0], color='blue', lw=2, label='IQR +/- Median')
whisker_legend = Line2D([0], [0], color='black', lw=1, label='Maximum range')

# Show only custom symbols (means, box, whisker), because lactation numbers are x-tick labels
plt.legend(handles=[mean_legend, box_legend, whisker_legend], 
           loc='best')
plt.grid()  
plt.tight_layout()
# plt.savefig("calving_intervals.tiff", dpi = 300)
plt.show()


In [ ]:
sim_list['days_in_milk'][0]

In [ ]:
sim_list['lactation_summary'][0]

In [ ]:
all_lact_summary = []
for df in sim_list['lactation_summary']:
    valid = df[df['max_lactation'] != 0]
    avg_lact = valid['max_lactation'].mean()
    std_lact = valid['max_lactation'].std()
    dim = []
    for dd in valid['days_in_milk_per_lactation']:
        for k, v in dd.items():
            # Exclude lactation 0 if present (usually dry-off)
            if k != 0:
                dim.append(v)
    if dim:
        median_dim = np.median(dim)
        q1_dim = np.percentile(dim, 25)
        q3_dim = np.percentile(dim, 75)
        iqr_dim = q3_dim - q1_dim
    else:
        median_dim, iqr_dim = np.nan, np.nan
    
    all_lact_summary.append({
        'average_lactation_number': avg_lact,
        'stddev_lactation_number': std_lact,
        'median_days_in_milk': median_dim,
        'iqr_days_in_milk': iqr_dim
    })

summary_df = pd.DataFrame(all_lact_summary)
# print(summary_df.describe())
summary_df